# 06.12 - Attention Mechanism

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Attention lets a model dynamically focus on different parts of the input when producing each output. Instead of compressing all info into a fixed hidden state, it creates direct connections between relevant positions.

## 2. Why Does This Matter?

Attention solved the information bottleneck of RNNs and is the foundation of transformers that dominate modern AI.

## 3. Prerequisites

- Unit 06.11 (RNNs/LSTMs), matrix multiplication + softmax

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Implement scaled dot-product attention from scratch
- Build multi-head attention and verify output shapes
- Explain Q/K/V, masking, positional encoding

## 5. Mental Model

Attention is a search engine: each output creates a Query ('what am I looking for?'), matches it against Keys ('what is available?'), and uses scores to weight Values ('what info to use').

```
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) · V
```


## 6. Backend


In [1]:
import matplotlib
matplotlib.use('Agg')
import math
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42); np.random.seed(42)
print("PyTorch version:", torch.__version__)


PyTorch version: 2.13.0+cpu


## 7. Scaled Dot-Product Attention from Scratch

Implement attention with optional masking and inspect the attention weights.


In [2]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights

# Random Q, K, V: batch=2, seq=4, d_k=3
Q = torch.randn(2, 4, 3); K = torch.randn(2, 4, 3); V = torch.randn(2, 4, 3)
out, w = scaled_dot_product_attention(Q, K, V)
print("Output shape:", tuple(out.shape))
print("Attention weights rows sum to 1:", torch.allclose(w.sum(dim=-1), torch.ones(2, 4), atol=1e-5))
print("Sample weight matrix (batch 0):\n", np.round(w[0].numpy(), 2))


Output shape: (2, 4, 3)
Attention weights rows sum to 1: True
Sample weight matrix (batch 0):
 [[0.11 0.06 0.02 0.81]
 [0.07 0.58 0.24 0.12]
 [0.3  0.1  0.56 0.04]
 [0.24 0.23 0.45 0.09]]


## 8. Why Scale by sqrt(d_k)?

Without scaling, large keys push softmax into a saturated regime where gradients vanish.


In [3]:
def softmax_scores(Q, K):
    raw = torch.matmul(Q, K.transpose(-2, -1))
    return torch.softmax(raw, dim=-1)

for d_k in [2, 16, 64]:
    q = torch.randn(1, 3, d_k); k = torch.randn(1, 3, d_k)
    w_scaled = torch.softmax(q @ k.transpose(-2,-1) / math.sqrt(d_k), dim=-1)
    w_raw    = softmax_scores(q, k)
    ent_scaled = -(w_scaled * torch.log(w_scaled + 1e-9)).sum(-1).mean().item()
    ent_raw    = -(w_raw    * torch.log(w_raw    + 1e-9)).sum(-1).mean().item()
    print(f"d_k={d_k:2d}: entropy(scaled)={ent_scaled:.3f}  entropy(unscaled)={ent_raw:.3f}")
print("\nScaling keeps the softmax from saturating (higher entropy = healthier gradients).")


d_k= 2: entropy(scaled)=0.907  entropy(unscaled)=0.790
d_k=16: entropy(scaled)=0.781  entropy(unscaled)=0.270
d_k=64: entropy(scaled)=0.913  entropy(unscaled)=0.446

Scaling keeps the softmax from saturating (higher entropy = healthier gradients).


## 9. Causal (Autoregressive) Masking

A lower-triangular mask prevents attending to future positions.


In [4]:
seq = 6
mask = torch.tril(torch.ones(seq, seq)).view(1, seq, seq)
Q = torch.randn(1, seq, 4); K = torch.randn(1, seq, 4); V = torch.randn(1, seq, 4)
out, w = scaled_dot_product_attention(Q, K, V, mask)
print("Causal mask (lower triangular):")
print(mask[0].int().tolist())
print("\nIn row 2 the model only attends to positions 0..2:")
print(np.round(w[0, 2].numpy(), 3))


Causal mask (lower triangular):
[[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 1]]

In row 2 the model only attends to positions 0..2:
[0.099 0.192 0.709 0.    0.    0.   ]


## 10. Multi-Head Attention

Parallel attention heads with learned projections; output shape equals input.


In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=16, num_heads=4):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    def forward(self, query, key, value, mask=None):
        b = query.size(0)
        Q = self.W_q(query).view(b, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(b, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(b, -1, self.num_heads, self.d_k).transpose(1, 2)
        out, _ = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(b, -1, self.d_model)
        return self.W_o(out)

mha = MultiHeadAttention(d_model=16, num_heads=4)
x = torch.randn(2, 6, 16)  # batch=2, seq=6, d_model=16
y = mha(x, x, x)
print("Multi-head self-attention output:", tuple(y.shape), "(same as input)")
print("Head dim d_k =", mha.d_k)


Multi-head self-attention output: (2, 6, 16) (same as input)
Head dim d_k = 4


## 11. Positional Encoding

Attention is position-agnostic, so we add sinusoidal position information.


In [6]:
def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    pos = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe.unsqueeze(0)

pe = positional_encoding(12, 16)
print("Positional encoding shape:", tuple(pe.shape))
print("Row for position 0:", np.round(pe[0, 0].numpy(), 3))
print("Row for position 5 differs from position 0:", not torch.allclose(pe[0,5], pe[0,0]))
print("\nDistinct per-position encodings let the model know sequence order.")


Positional encoding shape: (1, 12, 16)
Row for position 0: [0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.]
Row for position 5 differs from position 0: True

Distinct per-position encodings let the model know sequence order.


## 12. Common Mistakes & Debugging

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Nearly uniform weights | Not scaling; d_k large | Check scores | Add sqrt(d_k) scale |
| Attends to wrong positions | No positional encoding | Visualize weights | Add positional encoding |
| Memory on long seq | O(n^2) attention | Check length | Sparse/chunked attention |
| Generates from future | Missing causal mask | Check mask | Lower-triangular mask |

## 13. Real-World Considerations

- Machine translation: when generating 'chat', the decoder attends to 'cat'.
- Cross-attention: Q from decoder, K/V from encoder.

## 14. When NOT to Use

- Very long sequences where O(n^2) quadratic cost dominates — consider sparse attention.

## 15. Challenge

Verify a self-attention layer is permutation-equivariant: permuting input rows permutes output rows.


In [7]:
# Challenge: permutation equivariance
torch.manual_seed(3)
mha2 = MultiHeadAttention(d_model=8, num_heads=2)
x = torch.randn(1, 5, 8)
perm = torch.tensor([2, 0, 4, 1, 3])
out_full   = mha2(x, x, x)
out_perm   = mha2(x[:, perm], x[:, perm], x[:, perm])
print("Permuting inputs permutes outputs the same way:",
      bool(torch.allclose(out_perm[:, perm.argsort()], out_full, atol=1e-4)))
print("\nThis equivariance is why positional encoding is needed to break symmetry.")


Permuting inputs permutes outputs the same way: True

This equivariance is why positional encoding is needed to break symmetry.


## 16. Closed-Book Recall

Without looking back:

1. Why scale dot-product attention by sqrt(d_k)?
2. Difference between self-attention and cross-attention?
3. Why is positional encoding necessary?
4. Complexity of standard attention and why it's a problem?

## 17. Teach-Back Questions

Explain to another person:

- The Q/K/V roles in attention.
- What multi-head attention and causal masking do.

## 18. Summary

You implemented scaled dot-product attention, multi-head attention, causal masking, positional encoding, and verified equivariance.

## 19. Further Experiment

- Visualize attention weight heatmaps.
- Add a multi-head attention layer to the RNN classifier from Unit 06.11.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
